In [2]:
# --- 1. INSTALASI LIBRARY HUGGING FACE ---
!pip install transformers datasets accelerate

import os
from transformers import (
    BertTokenizer,
    BertForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    LineByLineTextDataset
)

# --- 2. SETUP MODEL DASAR (BASE MODEL) ---
# Torang pake IndoBERT punya IndoBenchmark karena basenya bahasa Indonesia (mirip Manado)
model_name = "indobenchmark/indobert-base-p1"

print(f"🚀 Sedang mendownload model dasar: {model_name}...")
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForMaskedLM.from_pretrained(model_name)

# --- 3. PERSIAPAN DATASET ---
print("📦 Sedang memproses dataset Manado jadi format angka (Tokenizing)...")

# Fungsi buat load dataset txt tadi
def load_dataset(file_path, tokenizer):
    return LineByLineTextDataset(
        tokenizer=tokenizer,
        file_path=file_path,
        block_size=128  # Panjang kalimat maksimal (biar hemat memori)
    )

train_dataset = load_dataset('data_train_manado.txt', tokenizer)
val_dataset = load_dataset('data_val_manado.txt', tokenizer)

# Data Collator: Ini yang tugasnya random tutup kata (Masking 15%)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)

# --- 4. SETUP TRAINING ARGUMENTS (Setelan Mesin) ---
training_args = TrainingArguments(
    output_dir="./indobert-manado-checkpoints", # Folder simpan sementara
    overwrite_output_dir=True,
    num_train_epochs=3,             # 3 putaran so cukup buat adaptasi awal
    per_device_train_batch_size=8,  # Jangan gede-gede biar GPU T4 nda jebol
    save_steps=500,                 # Simpan tiap 500 langkah
    save_total_limit=2,             # Cuma simpan 2 file terakhir biar disk nda penuh
    eval_strategy="epoch",    # Cek kepintaran tiap akhir putaran
    learning_rate=2e-5,             # Kecepatan belajar (pelan tapi pasti)
    weight_decay=0.01,
    report_to="none"                # Nda usah lapor ke wandb
)

# --- 5. EKSEKUSI TRAINING ---
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("\n🔥 MULAI TRAINING... (Sabar neh, kipas laptop mungkin mo bunyi kencang)")
trainer.train()

# --- 6. SIMPAN MODEL JADI ---
print("\n💾 Sedang menyimpan model final...")
save_path = "./model_indobert_manado_v1"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"🎉 SUKSES! Model tersimpan di folder: {save_path}")
print("Model ini sekarang so mengerti: 'Ngana', 'Torang', 'Kiapa'!")

🚀 Sedang mendownload model dasar: indobenchmark/indobert-base-p1...


Some weights of BertForMaskedLM were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


📦 Sedang memproses dataset Manado jadi format angka (Tokenizing)...

🔥 MULAI TRAINING... (Sabar neh, kipas laptop mungkin mo bunyi kencang)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,No log,3.707088
2,No log,2.336891
3,No log,2.151482


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



💾 Sedang menyimpan model final...
🎉 SUKSES! Model tersimpan di folder: ./model_indobert_manado_v1
Model ini sekarang so mengerti: 'Ngana', 'Torang', 'Kiapa'!


In [7]:
from transformers import pipeline

# Load model yang barusan torang latih
fill_mask = pipeline(
    "fill-mask",
    model="./model_indobert_manado_v1",
    tokenizer="./model_indobert_manado_v1"
)

# Tes Kalimat Manado (Tanda [MASK] adalah kata yang dia harus tebak)
kalimat_tes = "Ngana so [MASK] nasi goreng tadi malam?"

hasil = fill_mask(kalimat_tes)

print("🤖 Prediksi IndoBERT-Manado:")
for prediksi in hasil:
    print(f"Kata: {prediksi['token_str']} | Yakin: {prediksi['score']:.4f}")

Device set to use cpu


🤖 Prediksi IndoBERT-Manado:
Kata: jual | Yakin: 0.5284
Kata: cari | Yakin: 0.0081
Kata: so | Yakin: 0.0080
Kata: suka | Yakin: 0.0072
Kata: . | Yakin: 0.0060


In [4]:
# --- 1. LOAD MODEL YANG SO JADI TADI ---
# Bedanya di sini: Torang load dari FOLDER LOKAL, bukan dari Internet
checkpoint_path = "./model_indobert_manado_v1"

print(f"🔄 Melanjutkan training dari: {checkpoint_path}...")
model = BertForMaskedLM.from_pretrained(checkpoint_path)
tokenizer = BertTokenizer.from_pretrained(checkpoint_path)

# --- 2. SETUP TRAINING TAMBAHAN ---
training_args_lanjutan = TrainingArguments(
    output_dir="./indobert-manado-checkpoints-v2",
    overwrite_output_dir=True,
    num_train_epochs=5,             # Tambah 3 putaran lagi
    per_device_train_batch_size=8,
    save_steps=500,
    save_total_limit=2,
    eval_strategy="epoch",          # Ingat, pake eval_strategy (versi baru)
    learning_rate=1e-5,             # KITA TURUNKAN DIKIT (Biar belajar lebih teliti/halus)
    weight_decay=0.01,
    report_to="none"
)

# --- 3. EKSEKUSI LANJUTAN ---
trainer_lanjutan = Trainer(
    model=model,
    args=training_args_lanjutan,
    data_collator=data_collator,    # Pake data collator yang sama kayak tadi
    train_dataset=train_dataset,    # Pake dataset yang sama
    eval_dataset=val_dataset,
)

print("\n🔥 GASPOLL RONDE 2... (Semoga Loss turun ke 1.xx)")
trainer_lanjutan.train()

# --- 4. SIMPAN VERSI FINAL (V2) ---
save_path_v2 = "./model_indobert_manado_v2_final"
trainer_lanjutan.save_model(save_path_v2)
tokenizer.save_pretrained(save_path_v2)

print(f"\n🎉 MANTAP! Model V2 (yang lebe pintar) so tersimpan di: {save_path_v2}")

🔄 Melanjutkan training dari: ./model_indobert_manado_v1...

🔥 GASPOLL RONDE 2... (Semoga Loss turun ke 1.xx)


Epoch,Training Loss,Validation Loss
1,No log,1.935859
2,No log,1.679892
3,No log,1.609608
4,No log,1.656002
5,1.661900,1.336070


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



🎉 MANTAP! Model V2 (yang lebe pintar) so tersimpan di: ./model_indobert_manado_v2_final


In [14]:
from transformers import pipeline

# Load model yang barusan torang latih
fill_mask = pipeline(
    "fill-mask",
    model="./model_indobert_manado_v2_final", # Changed 'Final' to 'final'
    tokenizer="./model_indobert_manado_v2_final" # Changed 'Final' to 'final'
)

# Tes Kalimat Manado (Tanda [MASK] adalah kata yang dia harus tebak)
kalimat_tes = "So kenyang kt, Kt so [MASK] nasi goreng tadi"

hasil = fill_mask(kalimat_tes)

print("🤖 Prediksi IndoBERT-Manado:")
for prediksi in hasil:
    print(f"Kata: {prediksi['token_str']} | Yakin: {prediksi['score']:.4f}")

Device set to use cpu


🤖 Prediksi IndoBERT-Manado:
Kata: jual | Yakin: 0.8583
Kata: cari | Yakin: 0.0060
Kata: makan | Yakin: 0.0032
Kata: suka | Yakin: 0.0024
Kata: ada | Yakin: 0.0023
